# 💧 Phân Tích & Huấn Luyện Machine Learning - Smart Irrigation
Notebook này triển khai Pipeline Machine Learning có nhiệm vụ dự đoán lượng nước tưới tối ưu (`water_amount`). 

**Lưu ý quan trọng**: Do biến mục tiêu `water_amount` là dạng liên tục (Regression), các mô hình phân loại (Classification) sẽ được chuyển đổi sang dạng hồi quy tương ứng:
- Logistic Regression $\rightarrow$ Linear Regression / Ridge
- SVM $\rightarrow$ SVR (Support Vector Regressor)
- Random Forest $\rightarrow$ Random Forest Regressor
- XGBoost $\rightarrow$ XGBRegressor

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("✅ Các thư viện đã được import thành công.")

## 1. Tải và nạp Dữ liệu

In [ ]:
# Đọc dữ liệu từ thư mục dataset
df = pd.read_csv('dataset/smart_irrigation_dataset.csv')
print("Kích thước dataset:", df.shape)
df.head()

## 2. Tiền xử lý (Preprocessing & Feature Engineering)
- Mã hoá các cột loại cây, giai đoạn... (Categorical Features) thành dạng One-Hot Encoding
- Chuẩn hóa các Cột Số liệu điều kiện thời tiết (Numerical Features) thông qua StandardScaler.

In [ ]:
# Chia features và target
X = df.drop(columns=['water_amount'])
y = df['water_amount']

# Phân loại cột rễ chuẩn hóa
numeric_features = ['temperature', 'air_humidity', 'soil_moisture', 'light', 'rainfall']
categorical_features = ['soil_type', 'growth_stage', 'plant_type']

# Build Preprocessing pipeline steps
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("✅ Pipeline cấu hình tiền xử lý đã hoàn thành.")

## 3. Khởi tạo Model & Đánh giá bằng K-Fold Cross Validation
So sánh 4 Model theo chuẩn mà người dùng đã thiết lập.

In [ ]:
# Định nghĩa K-Fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Dictionary lưu model huấn luyện
models = {
    "Linear Regression": LinearRegression(),
    "SVR": SVR(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, objective='reg:squarederror')
}

results = {}

# Lưu ý: Việc train SVR với 15000 mẫu qua 5 folds có thể mất đôi chút thời gian
for name, model in models.items():
    print(f"⏳ Đang đánh giá {name}...")
    
    # Tạo pipeline Model cuối 
    model_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    
    # Chạy cross-validation
    cv_scores = cross_val_score(model_pipeline, X, y, cv=kf, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-cv_scores)
    results[name] = np.mean(rmse_scores)
    
    print(f"👉 {name} - RMSE trung bình: {np.mean(rmse_scores):.4f} (+/- {np.std(rmse_scores):.4f})\n")
    
best_model_name = min(results, key=results.get)
print(f"🏆 Mô hình tốt nhất trên tập CV là: {best_model_name}")

## 4. Huấn Luyện Full Model Tốt Nhất & Xuất File Pipeline (Model Object)
Sau khi tìm ra Model vượt trội nhất, chúng ta train lại nó toàn bộ và đóng gói luôn thuật toán Preprocessing vào chung 1 file (`.joblib`). Khi inference ở Backend, chỉ cần truyền data thô chưa OneHot và Standard.

In [ ]:
# Giả sử XGBoost hoặc Random Forest là Best, ta trực tiếp gán.
# Ở đây để đảm bảo an toàn nếu XGB tốt nhất:
best_algo = models[best_model_name]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Xây dựng full pipeline 
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('regressor', best_algo)
])

print(f"🔥 Bắt đầu train lại toàn bộ {best_model_name}...")
final_pipeline.fit(X_train, y_train)

# Đánh giá kiểm thử nhỏ trên tập Hold-out 20%
y_pred = final_pipeline.predict(X_test)
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test R2 Score: {r2_score(y_test, y_pred):.4f}")

# EXPORT
model_path = 'model_watering_prediction.joblib'
joblib.dump(final_pipeline, model_path)
print(f"✅ Pipeline gồm Preprocessing và Model đã được trọn gói tại {model_path}.")